## [통계적 검정과 인과추론] - A/B 테스트
### 주제 : 유료광고(cpc) vs 자연검색(organic) 유입 유저의 구매 전환율 비교 — 유료광고는 실제로 효과가 있는가?
### 작성자 : 한서연
### 작성일 : 2026.07.31
---
### 데이터셋설명 (1차 분석과 동일 데이터)
1. 이름 :  GA4 Obfuscated Sample E-commerce Dataset
2. 출처 : 구글이 운영하는 실제 온라인 쇼핑몰(Google Merchandise Store)의 방문 로그

### 수정사항
1. 확장된 표본으로 카이제곱 독립성 검정 재실시
2. Wilson 방법으로 95% 신뢰구간 재계산 
3. 검정력/MDE 재계산 
4. 로지스틱 회귀(`purchased ~ medium + device + is_new_visitor`)로 기기 종류·신규방문여부를 통제한 medium 효과 확인
---

### 1. 데이터 위생 점검 (2차 분석 — 3개월 확장)

| 항목 | 결과 | 1차(11월) 대비 |
|---|---|---|
| 결측치 | 핵심 컬럼 전부 0건 | 동일 (문제 없음) |
| 데이터 기간 | 20201101~20210131, 92일 | 11일 → 92일로 확장 확인 (11+31+31=92, 정상) |
| 완전 중복 | 없음 | 동일 (문제 없음) |
| cpc 전체 유저 | 15,528명 | 4,421명 → 약 3.5배 증가 |
| organic 전체 유저 | 112,358명 | 32,975명 → 약 3.4배 증가 |
| 겹치는 유저 | 1,630명 | 496명 → 절대 수는 늘었지만 비율은 비슷 |

- cpc 대비 겹침 비율: 1,630 / 15,528 ≈ 10.5% (1차: 11.2%)
- organic 대비 겹침 비율: 1,630 / 112,358 ≈ 1.5% (1차: 1.5%)

기간을 3개월로 확장했음에도 겹침 비율이 1차와 유사한 수준으로 유지됨을 확인함. 이에 따라 last-touch 기준 그룹 재정의 방식을 2차 분석에서도 동일하게 적용함.

**최종 데이터 (2차 분석 — 20201101~20210131, 3개월)**
|  | 구매 | 비구매 | 종합
|---|---|---|---|
| cpc | 232 | 14,152 | 14,384 |
| organic | 2,587 | 109,285 | 111,872 |

In [ ]:
# 데이터 기간 확인
SELECT
  MIN(_TABLE_SUFFIX) AS earliest_date,
  MAX(_TABLE_SUFFIX) AS latest_date,
  COUNT(DISTINCT _TABLE_SUFFIX) AS num_days
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
# 결과 : 시작일 : 2020-11-01 / 종료일 : 2021-01-31

# 전체 행 수 확인
SELECT COUNT(*) AS total_rows
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
# 결과 : 4295584	

# 결측치 확인 (핵심 컬럼)
SELECT
  COUNTIF(event_name IS NULL) AS null_event_name,
  COUNTIF(device.category IS NULL) AS null_device_category,
  COUNTIF(user_pseudo_id IS NULL) AS null_user_id
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'

# 데이터 날짜 범위 재확인
SELECT
  MIN(event_date) AS start_date,
  MAX(event_date) AS end_date,
  COUNT(DISTINCT event_date) AS num_days
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'

# 완전 중복(같은 유저 + 같은 시각 + 같은 이벤트) 확인
SELECT
  user_pseudo_id,
  event_timestamp,
  event_name,
  COUNT(*) AS cnt
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
GROUP BY user_pseudo_id, event_timestamp, event_name
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 20

# medium별 유저 수 확인 (1차에서 했던 것과 동일한 목적)
SELECT
  traffic_source.medium,
  COUNT(DISTINCT user_pseudo_id) AS user_count
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
GROUP BY traffic_source.medium
ORDER BY user_count DESC

SELECT
  traffic_source.medium,
  COUNT(DISTINCT user_pseudo_id) AS user_count
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
  AND traffic_source.medium IN ('cpc', 'organic')
GROUP BY traffic_source.medium

# 배정일관성 :  두 medium 모두에서 관측된 유저 수
SELECT COUNT(DISTINCT user_pseudo_id) AS overlapping_users
FROM (
  SELECT
    user_pseudo_id,
    COUNT(DISTINCT traffic_source.medium) AS medium_count
  FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
    AND traffic_source.medium IN ('cpc', 'organic')
  GROUP BY user_pseudo_id
  HAVING medium_count > 1
)

# last-touch
WITH last_touch AS (
  SELECT
    user_pseudo_id,
    traffic_source.medium AS medium,
    ROW_NUMBER() OVER (
      PARTITION BY user_pseudo_id 
      ORDER BY event_timestamp DESC
    ) AS rn
  FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
    AND traffic_source.medium IN ('cpc', 'organic')
)
SELECT
  medium,
  COUNT(*) AS user_count
FROM last_touch
WHERE rn = 1
GROUP BY medium

# 구매 여부 결합
WITH last_touch AS (
  SELECT
    user_pseudo_id,
    traffic_source.medium AS medium,
    ROW_NUMBER() OVER (
      PARTITION BY user_pseudo_id 
      ORDER BY event_timestamp DESC
    ) AS rn
  FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
    AND traffic_source.medium IN ('cpc', 'organic')
),
user_group AS (
  SELECT user_pseudo_id, medium
  FROM last_touch
  WHERE rn = 1
),
user_purchase AS (
  SELECT DISTINCT user_pseudo_id
  FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
    AND event_name = 'purchase'
)
SELECT
  g.medium,
  COUNT(DISTINCT g.user_pseudo_id) AS total_users,
  COUNT(DISTINCT p.user_pseudo_id) AS purchasers
FROM user_group g
LEFT JOIN user_purchase p
  ON g.user_pseudo_id = p.user_pseudo_id
GROUP BY g.medium

### 2. 질문과 가설 정하기 

- 귀무가설 $H_0$ : **유료광고 유입 유저와 자연검색 유입 유저의 구매 전환율은 차이가 없다**
- 대립가설 $H_1$ : **유료광고 유입 유저와 자연검색 유입 유저의 구매 전환율은 차이가 있다**
-  유의수준 $\alpha$ = 0.05 , **양측**을 결정합니다.

### 3. 검정 고르기 (1차와 동일)
- 이번 분석의 성공지표 역시, "구매 여부"(했다/안했다)로, 비율형 지표. 비교할 집단은 cpc, organic 두 개임.
- 비율 지표 + 2집단 비교 조건에 해당하므로, 카이제곱 독립성 검정을 사용했음.

### 4. 검정 수행하고 3종 세트로 보고
- 점추정 (차이)	얼마나 차이 나는가
- 95% 신뢰구간(Wilson)	그 추정이 얼마나 불확실한가
- 효과 크기	그 차이가 큰 차이인가

In [2]:
import numpy as np
from statsmodels.stats.proportion import confint_proportions_2indep
from scipy.stats import chi2_contingency

# 데이터 입력 (2차 분석, 3개월 확장 표본)
organic_purchase, organic_total = 2587, 111872
cpc_purchase, cpc_total = 232, 14384

# ===== 1. 점추정 (전환율 차이) =====
organic_rate = organic_purchase / organic_total
cpc_rate = cpc_purchase / cpc_total
diff = organic_rate - cpc_rate

print("=== 1. 점추정 ===")
print(f"organic 전환율: {organic_rate:.4%}")
print(f"cpc 전환율: {cpc_rate:.4%}")
print(f"전환율 차이(%p): {diff:.4%}")
print()

# ===== 2. 95% 신뢰구간 (Wilson 기반 = 'newcomb') =====
# 주의: statsmodels 최신 버전에서는 method='wilson'이 아니라 'newcomb'을 씀
# Newcombe 방법 = 각 그룹의 Wilson score 구간을 결합한 방식 (Wilson 계열)
ci_low, ci_upp = confint_proportions_2indep(
    count1=organic_purchase, nobs1=organic_total,
    count2=cpc_purchase, nobs2=cpc_total,
    method='newcomb'
)
print("=== 2. 95% 신뢰구간 (Wilson/Newcombe) ===")
print(f"전환율 차이의 95% 신뢰구간: [{ci_low:.4%}, {ci_upp:.4%}]")
print()

# 참고: 1차에서 썼던 Wald 방법과 비교
ci_low_wald, ci_upp_wald = confint_proportions_2indep(
    count1=organic_purchase, nobs1=organic_total,
    count2=cpc_purchase, nobs2=cpc_total,
    method='wald'
)
print(f"(참고) Wald 방법 신뢰구간: [{ci_low_wald:.4%}, {ci_upp_wald:.4%}]")
print()

# ===== 3. 효과크기 (카이제곱 검정 + Cramér's V) =====
absolute_diff = (organic_rate - cpc_rate) * 100
relative_diff = organic_rate / cpc_rate

table = np.array([
    [organic_purchase, organic_total - organic_purchase],
    [cpc_purchase, cpc_total - cpc_purchase]
])
chi2, p, dof, expected = chi2_contingency(table, correction=False)

n = table.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))

print("=== 3. 효과크기 ===")
print(f"organic 전환율: {organic_rate:.2%}, cpc 전환율: {cpc_rate:.2%}")
print(f"절대 차이: {absolute_diff:+.2f}%p")
print(f"상대 차이: {relative_diff:.2f}배")
print(f"χ²({dof}) = {chi2:.2f}, p = {p:.4g}, Cramér's V = {cramers_v:.3f}")

=== 1. 점추정 ===
organic 전환율: 2.3125%
cpc 전환율: 1.6129%
전환율 차이(%p): 0.6996%

=== 2. 95% 신뢰구간 (Wilson/Newcombe) ===
전환율 차이의 95% 신뢰구간: [0.4640%, 0.9127%]

(참고) Wald 방법 신뢰구간: [0.4756%, 0.9235%]

=== 3. 효과크기 ===
organic 전환율: 2.31%, cpc 전환율: 1.61%
절대 차이: +0.70%p
상대 차이: 1.43배
χ²(1) = 28.57, p = 9.021e-08, Cramér's V = 0.015


### 5️-1. 한 걸음 더 - 검정력/MDE 재계산 

- **Q1. 이 실험은 효과를 잡을 ’힘’이 있었나**
    - 80% 검정력으로 +1%p를 잡으려면 그룹당 몇 명이 필요한가?
    - MDE — 지금 표본으로 잡을 수 있는 가장 작은 효과는? 
- A. : 
    - 80% 검정력으로 +1%p 차이를 잡으려면 그룹당 약 3,201명이 필요했다. 2차 분석의 실제 cpc 표본(14,384명)은 이 조건을 충분히 초과하여 만족한다.
    - 현재 표본이 80% 검정력으로 감지할 수 있는 최소 효과(MDE)는 0.33%p였다. 
    - 1차 분석(11월, cpc 4,082명)에서는 MDE가 0.72%p로, 실제 관찰된 차이(0.35%p)보다 커서 "차이가 없다"와 "감지할 힘이 부족했다"를 구분할 수 없었다.
    - 그러나 2차 분석에서의 표본은 0.33%p 이상의 효과만 볼 수 있었고, 실제로 관찰된 차이(0.70%p)는 이 MDE(0.33%p)보다 크므로, 이번 분석에서 나온 유의한 결과(p < 0.001)는 통계적으로 감지할 충분한 힘을 갖춘 상태에서 나온 결과라고 할 수 있다.

In [3]:
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# 2차 분석 데이터 (3개월 확장 표본)
organic_purchase, organic_total = 2587, 111872
cpc_purchase, cpc_total = 232, 14384

organic_rate = organic_purchase / organic_total
cpc_rate = cpc_purchase / cpc_total

analysis = NormalIndPower()

# ===== 1. 80% 검정력으로 +1%p 차이를 잡으려면 그룹당 몇 명 필요한가 =====
baseline_rate = cpc_rate
target_rate = cpc_rate + 0.01  # +1%p

effect_size_1pp = proportion_effectsize(target_rate, baseline_rate)

required_n = analysis.solve_power(
    effect_size=effect_size_1pp,
    alpha=0.05,
    power=0.8,
    ratio=1,
    alternative='two-sided'
)

print("=== 1. 80% 검정력으로 +1%p 차이를 잡으려면 필요한 표본 수 ===")
print(f"그룹당 필요한 표본 수: {required_n:.0f}명")
print()

# ===== 2. 지금 표본(cpc)으로 잡을 수 있는 최소효과(MDE) =====
mde_effect_size = analysis.solve_power(
    nobs1=cpc_total,
    alpha=0.05,
    power=0.8,
    ratio=organic_total / cpc_total,
    alternative='two-sided'
)

def h_to_p2(p1, h):
    phi1 = 2 * np.arcsin(np.sqrt(p1))
    phi2 = phi1 + h
    return np.sin(phi2 / 2) ** 2

mde_rate = h_to_p2(cpc_rate, mde_effect_size)
mde_pp = (mde_rate - cpc_rate) * 100

print("=== 2. 현재 표본으로 감지 가능한 최소효과(MDE) ===")
print(f"cpc 표본 수: {cpc_total:,}명")
print(f"80% 검정력에서 감지 가능한 최소 효과(MDE): {mde_pp:.2f}%p")

=== 1. 80% 검정력으로 +1%p 차이를 잡으려면 필요한 표본 수 ===
그룹당 필요한 표본 수: 3201명

=== 2. 현재 표본으로 감지 가능한 최소효과(MDE) ===
cpc 표본 수: 14,384명
80% 검정력에서 감지 가능한 최소 효과(MDE): 0.33%p


### 5-2. 한 걸음 더 — 교란변수 통제

- **Q1. medium(cpc/organic) 효과는 다른 변수 때문에 생긴 착시가 아닌가**
    - device(기기 종류), is_new_visitor(신규/재방문 여부)가 같다고 가정해도 medium의 효과가 여전히 남아있는가?
    - 만약 medium의 효과가 없다면 다른 변수중 전환율에 영향을 주는 변수는 무엇인가?
    - 만약 medium의 효과가 없다면 왜 카이제곱 검정에서는 "medium 때문에 차이가 난다"고 착각했던 것인가?
- A. : 
    - cpc와 organic만 비교했을 땐 organic이 확실히 전환율이 높은 것처럼 보였는데, device(기기)와 is_new_visitor(신규/재방문)를 같이 통제하고 나니 medium의 효과가 통계적으로 유의하지 않게 사라졌다. 
    - 즉 "organic이 cpc보다 더 좋다"는 결론이 사실은 medium 자체 때문이 아니라 다른 이유 때문이었다.
    - is_new_visitor는 p<0.001로 극도로 유의하고, 오즈비가 0.085이 나왔다. 재방문자보다 구매 오즈가 91.5% 낮다는 뜻.
    - 이건 상식적으로 봐도, 이미 브랜드를 알고 다시 온 사람이 구매할 확률이 훨씬 높은 게 당연한 것 처럼 보인다.
    - device_category는 애매하게 유의해 보인다. mobile은 p=0.067로 관례적 유의수준(0.05)을 살짝 넘겨서 유의하지 않다고 보는 게 맞다. (아깝게 걸쳐있긴 하지만)
    - tablet은 완전히 유의하지 않았다. (p=0.971, 오즈비도 거의 1.00)
    - cpc 유입 유저의 92.3%가 신규 방문자인 반면, organic 유입 유저는 83.8%만 신규 방문자이다. 바꿔 말하면, organic에는 재방문자(16.2%)가 cpc(7.7%)보다 약 2.1배 더 많이 섞여 있다.
    - 다시 말해, 로지스틱 결과 재방문자는 신규 방문자보다 구매 확률이 11.8배나 높았다. 그러니까 organic 그룹 = 구매를 잘하는 재방문자가 상대적으로 많이 섞인 그룹이고, cpc 그룹 = 구매를 잘 못하는 신규 방문자가 상대적으로 많이 섞인 그룹이다.
    - 즉 1차와 2차 분석에서 관찰된 "organic > cpc 전환율 차이"는 실제로는 유입경로(medium) 자체의 효과가 아니라, 두 그룹에 섞여있는 신규/재방문자 비율이 달라서 생긴 교란 효과였다는 게 최종적으로 밝혀졌다.

In [ ]:
# 1단계: BigQuery 쿼리 (유저 단위 데이터 추출)
# 참고: "신규/재방문 여부"는 GA4 데이터에서 event_params의 ga_session_number 값으로 판단합니다.
# 이 값이 1이면 그 유저의 첫 세션(신규), 2 이상이면 재방문 세션입니다.
# 이번 분석에서는 last-touch(마지막 접촉) 이벤트 기준의 세션 번호를 사용하겠습니다.

WITH last_touch AS (
  SELECT
    user_pseudo_id,
    traffic_source.medium AS medium,
    device.category AS device_category,
    (SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_number') AS session_number,
    event_name,
    ROW_NUMBER() OVER (
      PARTITION BY user_pseudo_id
      ORDER BY event_timestamp DESC
    ) AS rn
  FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
    AND traffic_source.medium IN ('cpc', 'organic')
),
user_group AS (
  SELECT
    user_pseudo_id,
    medium,
    device_category,
    CASE WHEN session_number = 1 THEN 1 ELSE 0 END AS is_new_visitor
  FROM last_touch
  WHERE rn = 1
),
user_purchase AS (
  SELECT DISTINCT user_pseudo_id
  FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
    AND event_name = 'purchase'
)
SELECT
  g.user_pseudo_id,
  g.medium,
  g.device_category,
  g.is_new_visitor,
  IF(p.user_pseudo_id IS NOT NULL, 1, 0) AS purchased
FROM user_group g
LEFT JOIN user_purchase p
  ON g.user_pseudo_id = p.user_pseudo_id

In [8]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np

# CSV 불러오기
df = pd.read_csv('C:/DI/[노드4] 통계적 검정과 인과추론 (D012~D017)/D017/bquxjob_152f9c07_19fc5d0238e.csv')

# ===== 1. 데이터 기본 확인 =====
print("=== 데이터 기본 확인 ===")
print(df.shape)
print(df['medium'].value_counts())
print(df['device_category'].value_counts())
print(df['is_new_visitor'].value_counts())
print()

# ===== 2. 로지스틱 회귀 =====
model = smf.logit(
    formula='purchased ~ medium + device_category + is_new_visitor',
    data=df
).fit()

print(model.summary())
print()

# ===== 3. 오즈비 변환 =====
odds_ratios = np.exp(model.params)
conf = np.exp(model.conf_int())
conf.columns = ['2.5%', '97.5%']
result = pd.concat([odds_ratios, conf], axis=1)
result.columns = ['Odds Ratio', '2.5%', '97.5%']
print("=== 오즈비 ===")
print(result)

=== 데이터 기본 확인 ===
(126256, 5)
medium
organic    111872
cpc         14384
Name: count, dtype: int64
device_category
desktop    73180
mobile     50231
tablet      2845
Name: count, dtype: int64
is_new_visitor
1    107058
0     19198
Name: count, dtype: int64

Optimization terminated successfully.
         Current function value: 0.091862
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:              purchased   No. Observations:               126256
Model:                          Logit   Df Residuals:                   126251
Method:                           MLE   Df Model:                            4
Date:                Mon, 03 Aug 2026   Pseudo R-squ.:                  0.1412
Time:                        13:15:31   Log-Likelihood:                -11598.
converged:                       True   LL-Null:                       -13505.
Covariance Type:            nonrobust   LLR p-value:                     0.000
        

In [ ]:
# medium별 신규/재방문 비율
print(pd.crosstab(df['medium'], df['is_new_visitor'], normalize='index'))

is_new_visitor         0         1
medium                            
cpc             0.076682  0.923318
organic         0.161747  0.838253


## 6. 의사결정 마무리 (2차 분석)

### 1. 질문과 가설

**분석 질문**: 유료광고(cpc) 유입 유저와 자연검색(organic) 유입 유저 간, 구매 전환율에 차이가 있는가? — 유료광고는 실제로 효과가 있는가?

- H0 (귀무가설): cpc 유입 유저와 organic 유입 유저의 구매 전환율은 차이가 없다
- H1 (대립가설): cpc 유입 유저와 organic 유입 유저의 구매 전환율은 차이가 있다
- 유의수준: α = 0.05, 양측검정

### 2. 데이터 위생 점검

- 데이터: BigQuery `ga4_obfuscated_sample_ecommerce` (2020년 11월~2021년 1월, 3개월/92일)
- 핵심 컬럼(event_name, device.category, user_pseudo_id) 결측치: 없음
- 완전 중복 이벤트: 없음
- **배정 일관성 재확인**: 동일 유저가 cpc·organic 양쪽에서 관측된 경우 1,630명 (cpc 대비 약 10.5%) — 1차(11.2%)와 유사한 수준으로 확인되어, last-touch 기준 재정의 방식을 그대로 유지
- 최종 표본: organic 111,872명, cpc 14,384명 (1차 대비 약 3.4~3.5배 확대)

### 3. 검정 선택 근거

1차와 동일하게 성공지표는 "구매 여부"(비율형 지표), 비교 집단은 cpc·organic 2개이므로 **카이제곱 독립성 검정**을 기본 검정으로 사용하였다. 추가로 1차 한계를 보완하기 위해 신뢰구간 계산 방법(Wilson/Newcombe)과 교란변수를 통제한 **로지스틱 회귀**를 함께 사용하였다.

### 4. 결과

**카이제곱 검정 (확장 표본)**

| 항목 | 값 |
|---|---|
| organic 전환율 | 2.31% |
| cpc 전환율 | 1.61% |
| 절대 차이 | +0.70%p |
| 상대 차이 | 1.43배 |
| 95% 신뢰구간 (Newcombe/Wilson) | [0.46%p, 0.91%p] |
| χ²(1) | 28.57 |
| p-value | 9.02e-08 |
| Cramér's V | 0.015 |

1차와 달리, 표본을 3개월로 확대한 결과 organic과 cpc의 전환율 차이가 통계적으로 매우 유의하게 나타났다(p<0.001). 95% 신뢰구간도 0을 포함하지 않는다. 다만 Cramér's V(0.015)는 여전히 매우 작아, 통계적 유의성과 별개로 실질적인 효과크기는 작은 수준이다.

**검정력/MDE 분석**: 80% 검정력으로 +1%p 차이를 잡기 위해 필요한 표본 수는 그룹당 약 3,201명으로, 확장된 cpc 표본(14,384명)은 이를 충분히 초과한다. 현재 표본의 MDE는 0.33%p로 1차(0.72%p) 대비 크게 개선되었으며, 실제 관찰된 차이(0.70%p)가 MDE를 상회하여, 1차에서 미해결이었던 "차이가 없다 vs 감지력 부족"의 구분 불가 문제가 해소되었다.

**로지스틱 회귀 (교란변수 통제) — 핵심 발견**

| 변수 | 오즈비 | p-value |
|---|---|---|
| medium (organic) | 0.971 | 0.682 |
| device_category (mobile) | 1.076 | 0.067 |
| device_category (tablet) | 1.005 | 0.971 |
| is_new_visitor | 0.085 | <0.001 |

기기 종류와 신규/재방문 여부를 통제한 로지스틱 회귀(`purchased ~ medium + device_category + is_new_visitor`)에서는 **medium의 효과가 통계적으로 유의하지 않았다(p=0.682, 오즈비 0.97)**. 반면 is_new_visitor는 매우 강력하고 유의한 예측변수로(오즈비 0.085), 신규 방문자가 재방문자보다 구매 오즈가 약 91.5% 낮은 것으로 나타났다.

교차표 분석 결과, cpc 유입 유저의 92.3%가 신규 방문자인 반면 organic 유입 유저는 83.8%만 신규 방문자로, organic 그룹에 재방문자(구매 전환율이 원래 높은 집단)가 상대적으로 더 많이 섞여 있음을 확인하였다. 즉 **카이제곱에서 관찰된 organic-cpc 전환율 차이는 medium 자체의 효과가 아니라, 두 그룹의 신규/재방문자 구성비 차이에서 비롯된 교란효과**였다.

### 5. 의사결정

**→ 유료광고(cpc)가 자연검색(organic)보다 전환율이 낮다는 주장은 지지되지 않음**

- 표본을 확대한 카이제곱 검정만 보면 organic이 cpc보다 유의하게 높은 전환율을 보여, 얼핏 "유료광고 효과가 없다"는 결론으로 이어질 수 있어 보인다.
- 그러나 로지스틱 회귀로 교란변수(신규/재방문 여부, 기기 종류)를 통제한 결과, medium의 효과는 사라졌다(p=0.682). 관찰된 차이는 medium 자체가 아니라 두 채널의 유입 유저 구성(신규 vs 재방문 비율) 차이에서 비롯된 것으로 확인되었다.
- 따라서 "cpc가 organic보다 성과가 나쁘다"고 단정하여 예산을 재배분하는 것은 통계적으로 정당화되지 않는다.
- 오히려 실무적으로 더 유용한 결론은, **두 채널을 공정하게 비교하려면 신규 대 신규, 재방문 대 재방문으로 나누어 비교해야 한다**는 것이다. cpc는 태생적으로 신규 유저 획득 채널이므로 단순 전환율 비교만으로 채널 효과를 평가하면 오해를 낳을 수 있다.
- 다음 단계로는 신규 방문자 내에서만 medium별 전환율을 비교하는 하위분석을 진행할 것을 제안한다.

### 6. 한계

- **관찰연구라는 근본적 한계**: 1차와 동일하게, 이 분석은 무작위 배정 실험(RCT)이 아닌 관찰 데이터에 기반하므로, 통제된 변수 외에도 측정되지 않은 다른 특성(관심도, 구매의도 등)에 의한 자기선택 편향 가능성이 여전히 남아있다.
- **귀속방법의 한계**: 1차와 동일하게 last-touch 기준을 사용하였으며, 겹치는 유저 비율이 여전히 존재(cpc 대비 10.5%)하여 다른 귀속 기준(퍼스트터치, 선형귀속) 적용 시 결과가 달라질 수 있다.
- **일부 교란변수만 통제됨**: 기기 종류와 신규/재방문 여부만 통제하였으며, 지역, 방문 시간대 등 추가로 고려할 수 있는 교란변수는 분석의 규모를 생각했을 때, 현재 분석자의 레벨, 용량의 크기 등이 적절하지 않아 이번 분석 범위에서 제외하였다.
- **신뢰구간 방법 변경의 영향은 제한적**: Wald와 Newcombe(Wilson) 방법 간 신뢰구간 차이는 이번 표본 크기에서는 크지 않았다(각각 [0.48%p,0.92%p] vs [0.46%p,0.91%p]). 표본이 작았던 1차였다면 이 차이가 더 컸을 가능성이 있다.
- **상호작용 미검증**: medium과 is_new_visitor, medium과 device_category 간 상호작용항은 이번 모델에 포함하지 않았다. 즉 "신규 방문자 내에서는 cpc·organic 효과가 다를 수 있는가"와 같은 세부 질문은 확인하지 못했다.
- **단일 채널 한정**: Google Merchandise Store 데이터에 기반한 결과로, 다른 업종이나 플랫폼에 일반화하기 어렵다.